In [ ]:
"""
Comprehensive Multi File Strategy Analyzer
Option B aligned with Miroslav metric triad

Metric definitions

1 Authority Yield
  Mean relative PageRank change per inserted internal link
  Formula mean_delta_pct divided by k

2 Authority Volatility
  Standard deviation of Authority Yield across simulation runs
  We read std_delta from the CSV and normalize by k
  Formula std_delta divided by k

3 Down Up Ratio
  Expected pages_down divided by expected pages_up
  Formula avg_pages_down divided by max(avg_pages_up, 1)

Notes
  We do not recompute volatility from aggregated rows
  We read std_delta directly as produced by the experiment pipeline
  Strategy ordering and colors are fixed and reused across all plots using tab10
  Naming Convention: All strategies must use "[Name] candidates" format.
"""

from dataclasses import dataclass
from typing import List, Tuple, Optional
from enum import Enum
import os
import re
from datetime import datetime
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

try:
    from google.colab import drive

    COLAB_AVAILABLE = True
except ImportError:
    COLAB_AVAILABLE = False
    drive = None

warnings.filterwarnings("ignore")


# ============================================================
# Configuration and constants
# ============================================================

# UPDATED: Enforced strict "[Name] candidates" naming for all strategies
STRATEGY_ORDER = [
    "low candidates",
    "high candidates",
    "mixed candidates",
    "random candidates",
    "folder candidates",
]

TYPE_ORDER = ["automatic", "expert"]

TAB10 = mpl.colormaps["tab10"]
STRATEGY_COLORS = {s: TAB10(i % 10) for i, s in enumerate(STRATEGY_ORDER)}
TYPE_COLORS = {"automatic": TAB10(6), "expert": TAB10(7)}
DISTRIBUTION_COLORS = {"up": TAB10(2), "down": TAB10(3), "neutral": TAB10(7)}


class Config:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.initialize()
        return cls._instance

    def initialize(self):
        self.TOTAL_KALICUBE_PAGES = 1841
        self.INSERTED_LINKS_K = 240
        self.BASE_PATH = "/content/drive/MyDrive/WebKnoGraph"
        self.OUTPUT_PATH = "/content/drive/MyDrive/WebKnoGraph"
        self.DPI = 300
        self.STYLE = "seaborn-v0_8-whitegrid"


class FileType(Enum):
    AUTOMATIC = "automatic"
    EXPERT = "expert"
    OTHER = "other"


class SubCategory(Enum):
    BA = "ba"
    REAL_WWW = "real_www"
    UNKNOWN = "unknown"


@dataclass
class FileInfo:
    full_path: str
    relative_path: str
    filename: str
    folder: str
    size_kb: float
    modified: str
    type: FileType
    sub_category: SubCategory


# ============================================================
# File management
# ============================================================


class FileManager:
    def __init__(self):
        self.config = Config()
        self.mount_drive()

    def mount_drive(self):
        if COLAB_AVAILABLE:
            if not os.path.exists("/content/drive"):
                print("Mounting Google Drive")
                drive.mount("/content/drive", force_remount=True)
                print("Google Drive mounted\n")
            else:
                print("Google Drive already mounted\n")
        else:
            print("Google Colab not detected, skipping Drive mount")
            print(f"Ensure files are accessible at {self.config.BASE_PATH}\n")

    def categorize_file(self, file_path: str) -> Tuple[FileType, SubCategory]:
        path_lower = file_path.lower()

        if "automatic" in path_lower:
            type_category = FileType.AUTOMATIC
        elif "expert" in path_lower or "expert_led" in path_lower:
            type_category = FileType.EXPERT
        else:
            type_category = FileType.OTHER

        if "ba_" in path_lower or "/ba" in path_lower or "ba_results" in path_lower:
            sub_category = SubCategory.BA
        elif "real_www" in path_lower or "realwww" in path_lower:
            sub_category = SubCategory.REAL_WWW
        else:
            sub_category = SubCategory.UNKNOWN

        return type_category, sub_category

    def find_multi_files(self) -> List[FileInfo]:
        all_files: List[FileInfo] = []

        print("Searching for multi csv files")
        print("=" * 60)

        if not os.path.exists(self.config.BASE_PATH):
            print(f"Base path does not exist: {self.config.BASE_PATH}")
            return []

        for root, dirs, files in os.walk(self.config.BASE_PATH):
            dirs[:] = [d for d in dirs if not d.startswith(".") and "Trash" not in d]

            for file in files:
                if file.lower().startswith("multi") and file.lower().endswith(".csv"):
                    full_path = os.path.join(root, file)
                    relative_path = full_path.replace(self.config.BASE_PATH + "/", "")

                    try:
                        file_size = os.path.getsize(full_path)
                        mod_time = os.path.getmtime(full_path)
                        mod_date = datetime.fromtimestamp(mod_time).strftime(
                            "%Y-%m-%d %H:%M"
                        )

                        type_cat, sub_cat = self.categorize_file(relative_path)

                        all_files.append(
                            FileInfo(
                                full_path=full_path,
                                relative_path=relative_path,
                                filename=file,
                                folder=os.path.basename(os.path.dirname(full_path)),
                                size_kb=file_size / 1024,
                                modified=mod_date,
                                type=type_cat,
                                sub_category=sub_cat,
                            )
                        )
                    except Exception as e:
                        print(f"Error accessing file {file}: {e}")

        print(f"Found {len(all_files)} files\n")
        return all_files


# ============================================================
# Helper utilities
# ============================================================


def normalize_range_to_label(r: str) -> str:
    if r is None or (isinstance(r, float) and np.isnan(r)):
        return "unknown"
    s = str(r).strip()
    s = s.replace("–", "-").replace("--", "-").replace("_", "-")
    s = re.sub(r"\s+", "", s)

    nums = [int(x) for x in re.findall(r"\d+", s)]
    if len(nums) >= 2:
        return f"{nums[0]}-{nums[1]}"
    return s.lower() if s else "unknown"


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    mask = (~v.isna()) & (~w.isna()) & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float(np.average(v[mask], weights=w[mask]))


def group_weighted_mean(
    df: pd.DataFrame, value_col: str, weight_col: str = "total_simulations"
) -> float:
    if value_col not in df.columns:
        return np.nan
    if weight_col not in df.columns:
        return float(pd.to_numeric(df[value_col], errors="coerce").mean())
    return weighted_mean(df[value_col], df[weight_col])


# ============================================================
# Visualization
# ============================================================


class UnifiedVisualizer:
    def __init__(self):
        self.config = Config()
        self.figures: List[plt.Figure] = []
        try:
            plt.style.use(self.config.STYLE)
        except OSError:
            plt.style.use("ggplot")

    def create_all_plots(self, df: pd.DataFrame):
        if df.empty:
            print("DataFrame is empty, skipping plots")
            return

        sns.set_context("notebook", font_scale=1.2)
        sns.set_style("whitegrid")

        print("\nGenerating plots using Miroslav metric triad, Option B")
        print("=" * 60)

        self._create_strategy_performance_plots(df)
        self._create_comparative_plots(df)

        print(f"Created {len(self.figures)} figures")
        for i, fig in enumerate(self.figures, 1):
            print(f"Displaying figure {i}")
            plt.show()

    def _create_strategy_performance_plots(self, df: pd.DataFrame):
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        fig.suptitle(
            "Strategy performance using the metric triad",
            fontsize=18,
            fontweight="bold",
        )

        df = df.copy()
        if "total_simulations" not in df.columns:
            df["total_simulations"] = 1.0

        summary = (
            df.groupby("strategy", observed=True)
            .apply(
                lambda g: pd.Series(
                    {
                        "yield_mean": weighted_mean(
                            g["authority_yield"], g["total_simulations"]
                        ),
                        "volatility_mean": weighted_mean(
                            g["authority_volatility"], g["total_simulations"]
                        ),
                        "avg_up": weighted_mean(
                            g["avg_pages_up"], g["total_simulations"]
                        ),
                        "avg_down": weighted_mean(
                            g["avg_pages_down"], g["total_simulations"]
                        ),
                        "avg_neutral": weighted_mean(
                            g["avg_pages_neutral"], g["total_simulations"]
                        ),
                        "down_up": weighted_mean(
                            g["down_up_ratio"], g["total_simulations"]
                        ),
                    }
                )
            )
            .reset_index()
        )

        summary["strategy"] = pd.Categorical(
            summary["strategy"], categories=STRATEGY_ORDER, ordered=True
        )
        summary = summary.sort_values("strategy")

        x = np.arange(len(summary))
        colors = [
            STRATEGY_COLORS.get(s, TAB10(0)) for s in summary["strategy"].astype(str)
        ]

        # Top left
        ax = axes[0, 0]
        y = summary["yield_mean"].to_numpy(dtype=float)
        yerr = summary["volatility_mean"].to_numpy(dtype=float)
        ax.bar(x, y, color=colors)
        ax.errorbar(x, y, yerr=yerr, fmt="none", capsize=4, ecolor="black", alpha=0.5)
        ax.set_title("1 Authority Yield with Authority Volatility error bars")
        ax.set_ylabel("Authority Yield, mean Delta PR percent per inserted link")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=45, ha="right")

        # Top right
        ax = axes[0, 1]
        up = summary["avg_up"].to_numpy(dtype=float)
        down = summary["avg_down"].to_numpy(dtype=float)
        neutral = summary["avg_neutral"].to_numpy(dtype=float)

        if np.all(np.isnan(up)) or np.all(np.isnan(down)):
            ax.text(
                0.5,
                0.5,
                "Page distribution data missing",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_axis_off()
        else:
            if np.all(np.isnan(neutral)):
                neutral = self.config.TOTAL_KALICUBE_PAGES - (
                    np.nan_to_num(up) + np.nan_to_num(down)
                )

            neutral = np.maximum(neutral, 0)

            ax.bar(x, up, label="Up", color=DISTRIBUTION_COLORS["up"])
            ax.bar(
                x,
                down,
                bottom=np.nan_to_num(up),
                label="Down",
                color=DISTRIBUTION_COLORS["down"],
            )
            ax.bar(
                x,
                neutral,
                bottom=np.nan_to_num(up) + np.nan_to_num(down),
                label="Neutral",
                color=DISTRIBUTION_COLORS["neutral"],
            )
            ax.set_title("Page distribution")
            ax.set_ylabel(f"Average pages, total {self.config.TOTAL_KALICUBE_PAGES}")
            ax.set_xticks(x)
            ax.set_xticklabels(summary["strategy"].astype(str), rotation=45, ha="right")
            ax.legend()

        # Bottom left
        ax = axes[1, 0]
        ax.bar(x, summary["down_up"].to_numpy(dtype=float), color=colors)
        ax.set_title("3 Down Up Ratio")
        ax.set_ylabel("Expected Down divided by expected Up")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=45, ha="right")

        # Bottom right
        ax = axes[1, 1]
        ax.bar(x, summary["volatility_mean"].to_numpy(dtype=float), color=colors)
        ax.set_title("2 Authority Volatility")
        ax.set_ylabel("Std Dev of Authority Yield per inserted link")
        ax.set_xticks(x)
        ax.set_xticklabels(summary["strategy"].astype(str), rotation=45, ha="right")

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        self.figures.append(fig)

    def _create_comparative_plots(self, df: pd.DataFrame):
        fig, axes = plt.subplots(2, 2, figsize=(20, 14))
        fig.suptitle(
            "Comparative analysis across ranges and selection types",
            fontsize=18,
            fontweight="bold",
        )

        df = df.copy()
        df["range_label"] = df["range"].apply(normalize_range_to_label)

        # Top left
        ax = axes[0, 0]
        if "type" in df.columns and df["type"].notna().any():
            sns.boxplot(
                data=df,
                x="range_label",
                y="authority_yield",
                hue="type",
                hue_order=TYPE_ORDER,
                palette=TYPE_COLORS,
                ax=ax,
            )
            ax.legend(title="Type")
        else:
            sns.boxplot(data=df, x="range_label", y="authority_yield", ax=ax)
        ax.set_title("Authority Yield by connection range and selection type")
        ax.set_ylabel("Authority Yield, mean Delta PR percent per inserted link")

        # Top right
        ax = axes[0, 1]
        perf = (
            df.groupby(["range_label", "strategy"], observed=True)
            .apply(lambda g: group_weighted_mean(g, "authority_yield"))
            .reset_index(name="authority_yield")
            .dropna()
        )
        if not perf.empty:
            best_idx = perf.groupby("range_label", observed=True)[
                "authority_yield"
            ].idxmax()
            best = perf.loc[best_idx].copy()
            best["strategy"] = pd.Categorical(
                best["strategy"], categories=STRATEGY_ORDER, ordered=True
            )
            best = best.sort_values(["range_label", "strategy"])

            sns.barplot(
                data=best,
                x="range_label",
                y="authority_yield",
                hue="strategy",
                hue_order=STRATEGY_ORDER,
                palette=STRATEGY_COLORS,
                dodge=False,
                ax=ax,
            )
            ax.set_title("Highest yield strategy per connection range")
            ax.set_ylabel("Authority Yield, mean Delta PR percent per inserted link")
            ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        else:
            ax.set_axis_off()

        # Bottom left
        ax = axes[1, 0]
        if "type" in df.columns and df["type"].notna().any():
            sns.violinplot(
                data=df,
                x="range_label",
                y="down_up_ratio",
                hue="type",
                hue_order=TYPE_ORDER,
                palette=TYPE_COLORS,
                split=True,
                ax=ax,
            )
        else:
            sns.violinplot(data=df, x="range_label", y="down_up_ratio", ax=ax)
        ax.set_title("Down Up Ratio by connection range")
        ax.set_ylabel("Expected Down divided by expected Up")

        # Bottom right
        ax = axes[1, 1]
        scatter = (
            df.groupby(["strategy", "type", "range_label"], observed=True)
            .apply(
                lambda g: pd.Series(
                    {
                        "authority_yield": group_weighted_mean(g, "authority_yield"),
                        "down_up_ratio": group_weighted_mean(g, "down_up_ratio"),
                    }
                )
            )
            .reset_index()
        )

        sns.scatterplot(
            data=scatter,
            x="authority_yield",
            y="down_up_ratio",
            hue="strategy",
            style="type" if "type" in scatter.columns else None,
            alpha=0.75,
            palette=STRATEGY_COLORS,
            hue_order=STRATEGY_ORDER,
            ax=ax,
        )
        ax.set_title("Tradeoff between Authority Yield and Down Up Ratio")
        ax.set_xlabel("Authority Yield, higher is better")
        ax.set_ylabel("Down Up Ratio, lower is better")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        self.figures.append(fig)

    def save_figures(self, output_path: Optional[str] = None):
        if not output_path:
            output_path = self.config.OUTPUT_PATH

        plot_names = [
            "triad_strategy_performance",
            "triad_comparative_analysis",
        ]

        for fig, name in zip(self.figures, plot_names):
            plot_path = f"{output_path}/{name}.png"
            fig.savefig(plot_path, dpi=self.config.DPI, bbox_inches="tight")
            print(f"Saved {name} to {plot_path}")

        pdf_path = f"{output_path}/final_metric_triad_analysis.pdf"
        with PdfPages(pdf_path) as pdf:
            for fig in self.figures:
                pdf.savefig(fig, bbox_inches="tight")
        print(f"Saved PDF to {pdf_path}")


# ============================================================
# App
# ============================================================


class StrategyAnalyzerApp:
    def __init__(self):
        self.config = Config()
        self.file_manager = FileManager()
        self.visualizer = UnifiedVisualizer()

    def run(self):
        print("Starting analysis, Option B")

        all_files = self.file_manager.find_multi_files()
        if not all_files:
            print("No multi csv files found")
            return None

        master_df = self._load_and_combine_data(all_files)
        if master_df.empty:
            print("No data could be loaded")
            return None

        self._print_validation(master_df)
        self.visualizer.create_all_plots(master_df)

        if self._ask_to_save("plots"):
            self.visualizer.save_figures()

        if self._ask_to_save("results"):
            csv_path = os.path.join(self.config.OUTPUT_PATH, "master_triad_results.csv")
            master_df.to_csv(csv_path, index=False)
            print(f"Saved results to {csv_path}")

        return master_df

    def _load_and_combine_data(self, all_files: List[FileInfo]) -> pd.DataFrame:
        print("\nLoading and combining files")

        all_dfs: List[pd.DataFrame] = []
        # UPDATED: Mapping all inputs to strict "[Name] candidates" format
        strategy_mapping = {
            "low": "low candidates",
            "low candidates": "low candidates",
            "worst": "low candidates",
            "worst candidates": "low candidates",
            "high": "high candidates",
            "high candidates": "high candidates",
            "best": "high candidates",
            "best candidates": "high candidates",
            "mixed": "mixed candidates",
            "mixed candidates": "mixed candidates",
            "random": "random candidates",
            "random candidates": "random candidates",
            "folder": "folder candidates",
            "folder candidates": "folder candidates",
        }

        k = float(self.config.INSERTED_LINKS_K)

        for file_info in all_files:
            try:
                df = pd.read_csv(file_info.full_path)
                df["type"] = file_info.type.value
                df["sub_category"] = file_info.sub_category.value
                df["source_file"] = file_info.filename

                # Strategy normalization
                if "strategy" in df.columns:
                    df["strategy"] = df["strategy"].astype(str).str.lower().str.strip()
                    df["strategy"] = df["strategy"].replace(strategy_mapping)

                # Range normalization
                if "range" in df.columns:
                    df["range"] = df["range"].astype(str).str.strip()
                else:
                    df["range"] = "unknown"

                # Total simulations weight
                if "total_simulations" in df.columns:
                    df["total_simulations"] = pd.to_numeric(
                        df["total_simulations"], errors="coerce"
                    ).fillna(1.0)
                else:
                    df["total_simulations"] = 1.0

                # Authority Yield per inserted link
                raw_mean = None
                if "mean_delta_pct" in df.columns:
                    raw_mean = pd.to_numeric(df["mean_delta_pct"], errors="coerce")
                elif "delta_pr_percent_mean" in df.columns:
                    raw_mean = pd.to_numeric(
                        df["delta_pr_percent_mean"], errors="coerce"
                    )

                df["authority_yield"] = (
                    (raw_mean / k) if raw_mean is not None else np.nan
                )

                # Authority Volatility per inserted link
                raw_std = None
                if "std_delta" in df.columns:
                    raw_std = pd.to_numeric(df["std_delta"], errors="coerce")
                elif "delta_pr_percent_std" in df.columns:
                    raw_std = pd.to_numeric(df["delta_pr_percent_std"], errors="coerce")

                df["authority_volatility"] = (
                    (raw_std / k) if raw_std is not None else np.nan
                )

                # Down Up Ratio
                if "avg_pages_down" in df.columns and "avg_pages_up" in df.columns:
                    down = pd.to_numeric(df["avg_pages_down"], errors="coerce")
                    up = pd.to_numeric(df["avg_pages_up"], errors="coerce")
                    df["down_up_ratio"] = down / np.maximum(up, 1.0)
                else:
                    df["down_up_ratio"] = np.nan

                # Neutral pages context
                if "avg_pages_up" in df.columns and "avg_pages_down" in df.columns:
                    up = pd.to_numeric(df["avg_pages_up"], errors="coerce")
                    down = pd.to_numeric(df["avg_pages_down"], errors="coerce")
                    neutral = self.config.TOTAL_KALICUBE_PAGES - (up + down)
                    df["avg_pages_neutral"] = np.maximum(neutral, 0)
                else:
                    df["avg_pages_neutral"] = np.nan

                all_dfs.append(df)

            except Exception as e:
                print(f"Could not process {file_info.filename}: {e}")

        if not all_dfs:
            return pd.DataFrame()

        master_df = pd.concat(all_dfs, ignore_index=True)

        # Categorical ordering with fallback for unknown strategies
        if "strategy" in master_df.columns:
            current_strategies = master_df["strategy"].dropna().unique().tolist()
            unknown_strategies = [
                s for s in current_strategies if s not in STRATEGY_ORDER
            ]
            if unknown_strategies:
                for s in unknown_strategies:
                    STRATEGY_ORDER.append(s)
                    STRATEGY_COLORS[s] = TAB10(len(STRATEGY_ORDER) % 10)

            master_df["strategy"] = pd.Categorical(
                master_df["strategy"], categories=STRATEGY_ORDER, ordered=True
            )

        if "type" in master_df.columns:
            master_df["type"] = pd.Categorical(
                master_df["type"], categories=TYPE_ORDER, ordered=True
            )

        print(f"Combined {len(all_dfs)} files into {len(master_df)} rows")
        return master_df

    def _print_validation(self, df: pd.DataFrame):
        print("\nData validation")
        print("=" * 60)

        if "strategy" in df.columns:
            print("Strategies found")
            print(df["strategy"].dropna().unique().tolist())

        cols = ["authority_yield", "authority_volatility", "down_up_ratio"]
        existing = [c for c in cols if c in df.columns]
        if existing and "strategy" in df.columns:
            stats = df.groupby("strategy", observed=True)[existing].mean()
            print("\nMean metrics per strategy")
            print(stats)

        if "authority_yield" in df.columns:
            print(f"\nMissing yield {df['authority_yield'].isna().mean():.2%}")
        if "authority_volatility" in df.columns:
            print(f"Missing volatility {df['authority_volatility'].isna().mean():.2%}")

        print("=" * 60 + "\n")

    def _ask_to_save(self, item: str) -> bool:
        response = input(f"\nSave {item} to Drive (y or n): ").strip().lower()
        return response == "y"


def main():
    app = StrategyAnalyzerApp()
    results_df = app.run()

    if results_df is not None and not results_df.empty:
        print("\n" + "=" * 60)
        print("Analysis complete")
        print("=" * 60)

    return results_df


if __name__ == "__main__":
    master_results = main()